In [1]:
import pandas as pd
import numpy as np
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.preprocessing import OrdinalEncoder
from sklearn.preprocessing import MinMaxScaler
from sklearn.tree import DecisionTreeClassifier

In [2]:
df = pd.read_csv(r"C:\Users\karth\Downloads\train (1).csv")

Plan for sklearn pipeline "Column Transformer"

1) Transformer is off missing values 
2) Ohe 
3) scaling
4) feature selection 
5) Algorithm (in our case DT)


In [4]:
df.columns

Index(['PassengerId', 'Survived', 'Pclass', 'Name', 'Sex', 'Age', 'SibSp',
       'Parch', 'Ticket', 'Fare', 'Cabin', 'Embarked'],
      dtype='object')

In [5]:
df.drop(columns=['PassengerId','Name','Ticket','Cabin'],inplace=True)

In [6]:
from sklearn.model_selection import train_test_split
X_train,X_test,y_train,y_test = train_test_split(df.drop(columns=['Survived']),df['Survived'],test_size=0.2,random_state=42)

In [7]:
X_train.head(5)

,Pclass,Sex,Age,SibSp,Parch,Fare,Embarked
331,1,male,45.5,0,0,28.5000,S
733,2,male,23.0,0,0,13.0000,S
382,3,male,32.0,0,0,7.9250,S
704,3,male,26.0,1,0,7.8542,S
813,3,female,6.0,4,2,31.2750,S


In [8]:
y_train

331    0
733    0
382    0
704    0
813    0
      ..
106    1
270    0
860    0
435    1
102    0
Name: Survived, Length: 712, dtype: int64

In [10]:
# Imputation Transformer 
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline,make_pipeline
from sklearn.feature_selection import SelectKBest,chi2

## Instead of giving feature name we have given index of age column [2],we did
## that because transformer gives numpy array instead of pd dataframe 
## so when we will give missing transformer output to next transformer it will
# run smoothly as numpy array has no column name. if given name column it will give error when given to next transformer  

trf1 = ColumnTransformer(
    [('impute_age',SimpleImputer(),[2]),
     ('impute_embarked',SimpleImputer(strategy='most_frequent'),[6])],
     remainder='passthrough')


In [12]:
## ohe transformer not using drop first as i am using decision tree and it has no impact of multicollinearity

trf2 = ColumnTransformer(
    [('ohe_sex_embarked',OneHotEncoder(sparse_output=False,handle_unknown='ignore'),[1,6])],
    remainder='passthrough'
)


In [13]:
## scaling transformer 
## after applying trf2 transformer of ohe we will get 2 columns of sex and 3 columns of embarked . previous total (7)-(2)sex.embark col = 5 
# and new sex and embark column equal 5 . total column that our next scaling transformer will get is 10 as also we are not doing drop first
# so this slice(0,10) function will apply scaling(minmaxscaler in our case) in all column index from 0 to 10. 0 included and 10 excluded  

trf3 = ColumnTransformer([
    ('Scale',MinMaxScaler(),slice(0,10))
])

In [22]:
## Feature Selection
trf4 = SelectKBest(score_func=chi2,k=8)

In [17]:
## Train the model 
trf5 = DecisionTreeClassifier()

### Make Pipeline

In [23]:
pipe = Pipeline([('trf1',trf1),
                 ('trf2',trf2),
                 ('trf3',trf3),
                 ('trf4',trf4),
                 ('trf5',trf5)
])

## Pipeline vs Make_Pipeline 
Pipeline is a class which requires naming of steps whereas make_pipeline is a function which does not requires naming etc.
Same applies for the ColumnTransformer (Class) and make_column_tranformer (Function) so in make_column_tarnsformer you only send function and columne name . for eg. make_column_tranformer(SimpleImputer(),Age) doesnt have to give name .ColumnTransformer('impute_age',SimpleImputer(),Age)

In [20]:
## Alternate Syntax 
pipe_1 = make_pipeline(trf1,trf2,trf3,trf4,trf5)

In [24]:
# Train data 
pipe.fit(X_train,y_train)

Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('Scale', MinMaxScaler(),
                                                  slice(0, 10, None))])),
                ('trf4',
                 SelectKBest(k=8,
                             score_func=<function chi2 at 0x0000019980B8B9D0>)),
                ('trf5', DecisionTreeClassifier())])

In [30]:
pipe.named_steps

{'trf1': ColumnTransformer(remainder='passthrough',
                   transformers=[('impute_age', SimpleImputer(), [2]),
                                 ('impute_embarked',
                                  SimpleImputer(strategy='most_frequent'),
                                  [6])]),
 'trf2': ColumnTransformer(remainder='passthrough',
                   transformers=[('ohe_sex_embarked',
                                  OneHotEncoder(handle_unknown='ignore',
                                                sparse_output=False),
                                  [1, 6])]),
 'trf3': ColumnTransformer(transformers=[('Scale', MinMaxScaler(), slice(0, 10, None))]),
 'trf4': SelectKBest(k=8, score_func=<function chi2 at 0x0000019980B8B9D0>),
 'trf5': DecisionTreeClassifier()}

In [39]:
pipe.named_steps.trf1.transformers_[0][1].statistics_

## This gives you an information that age missing vakue is iummpute by mean of 29.498

array([29.49884615])

In [47]:
pipe.named_steps.trf1.transformers_[1][1].statistics_

array(['S'], dtype=object)

In [64]:
pipe.named_steps.trf3.transformers_[0][1].feature_range

(0, 1)

In [31]:
pipe.n_features_in_

7

In [65]:
pipe.get_feature_names_out

<bound method Pipeline.get_feature_names_out of Pipeline(steps=[('trf1',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('impute_age', SimpleImputer(),
                                                  [2]),
                                                 ('impute_embarked',
                                                  SimpleImputer(strategy='most_frequent'),
                                                  [6])])),
                ('trf2',
                 ColumnTransformer(remainder='passthrough',
                                   transformers=[('ohe_sex_embarked',
                                                  OneHotEncoder(handle_unknown='ignore',
                                                                sparse_output=False),
                                                  [1, 6])])),
                ('trf3',
                 ColumnTransformer(transformers=[('Scale', MinMaxScaler(),
                   

In [26]:
## Above visual diagram will be seen if we give code written below but in vscode it is already set not needed if using jupyter notebook than use
## Display Pipeline


# Code
# from sklearn import set_config
# set_config(display='diagram')

In [27]:
# Predict

y_pred = pipe.predict(X_test)

In [66]:
y_pred

array([1, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 0, 1, 0, 0, 0, 0, 0,
       1, 0, 0, 0, 0, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 0, 0,
       0, 0, 1, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0, 0, 1, 0, 1, 1, 1, 1,
       0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 0, 0, 0, 0, 0, 1,
       0, 0, 1, 0, 0, 0, 1, 1, 0, 0, 0, 1, 0, 0, 0, 1, 0, 1, 1, 0, 0, 1,
       0, 0, 1, 1, 1, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 0, 1, 0, 1, 0, 0, 0,
       0, 0, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1, 0, 0, 0, 0, 1, 0, 0, 1, 0, 1,
       0, 0, 0], dtype=int64)

In [28]:
from sklearn.metrics import accuracy_score
accuracy_score(y_test,y_pred)

0.6256983240223464

### Cross Validation using Pipeline

In [67]:
# cross validation using cross_val_score
from sklearn.model_selection import cross_val_score
cross_val_score(pipe, X_train, y_train, cv=5, scoring='accuracy').mean()

0.6391214419383433

### GridSearchCV using pipeline

Hyperparameter Tuning - By tuning parameters we can improve the performance of the model. parameter tuning may improve or may not improb=ve the model performance 

In decision tree there is max_depth parameter by tuning which we may get improved model performance 

In [68]:
# gridsearchcv
## TRF5 is the tranformer of decision tree model from which we are using DT parameter called max_depth
params = {
    'trf5__max_depth':[1,2,3,4,5,None]
}

In [69]:
from sklearn.model_selection import GridSearchCV
grid = GridSearchCV(pipe, params, cv=5, scoring='accuracy')
grid.fit(X_train, y_train)


GridSearchCV(cv=5,
             estimator=Pipeline(steps=[('trf1',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('impute_age',
                                                                         SimpleImputer(),
                                                                         [2]),
                                                                        ('impute_embarked',
                                                                         SimpleImputer(strategy='most_frequent'),
                                                                         [6])])),
                                       ('trf2',
                                        ColumnTransformer(remainder='passthrough',
                                                          transformers=[('ohe_sex_embarked',
                                                                         OneHotEncoder(handle_unknown='ignore',
                                                                                       sparse_output=False),
                                                                         [1,
                                                                          6])])),
                                       ('trf3',
                                        ColumnTransformer(transformers=[('Scale',
                                                                         MinMaxScaler(),
                                                                         slice(0, 10, None))])),
                                       ('trf4',
                                        SelectKBest(k=8,
                                                    score_func=<function chi2 at 0x0000019980B8B9D0>)),
                                       ('trf5', DecisionTreeClassifier())]),
             param_grid={'trf5__max_depth': [1, 2, 3, 4, 5, None]},
             scoring='accuracy')

In [70]:
grid.best_score_

0.6391214419383433

In [71]:

grid.best_params_

{'trf5__max_depth': 2}

### Exporting the pipeline

In [72]:
# Export 
import pickle

In [82]:
pickle.dump(pipe,open('models/pipe.pkl','wb'))

##### Everything is inside the pipe . we dont have to dump every column transformer separately as we have all transformer inside the pipeline 

In [83]:
import numpy as np
import pickle 

In [84]:
pipe = pickle.load(open('models/pipe.pkl','rb'))

In [86]:
X_train.columns

Index(['Pclass', 'Sex', 'Age', 'SibSp', 'Parch', 'Fare', 'Embarked'], dtype='object')

In [98]:
## Lets assume we have a user on website who is giving details to check for survival rate 

# test_input = np.array([2,'male',31.0,0,0,10.5,'S'],dtype=object).reshape(1,7)
test_input = np.array([2,'male',31.0,0,0,10.5,'S'],dtype=object).reshape(1,7)

In [99]:
test_input

array([[2, 'male', 31.0, 0, 0, 10.5, 'S']], dtype=object)

In [100]:
pipe.predict(test_input)

c:\Users\karth\anaconda3\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(
c:\Users\karth\anaconda3\lib\site-packages\sklearn\base.py:439: UserWarning: X does not have valid feature names, but SimpleImputer was fitted with feature names
  warnings.warn(


array([0], dtype=int64)

##### Well managed codes because of pipelines. when your code is in production you only need to export the pipe.pkl file and load into production .no change in code